### Libraries

In [ ]:
library(Seurat)
library(DESeq2)
library(readr)
library(dplyr)
library(tibble)
library(readxl)
library(pheatmap)
library(writexl)
library(ggplot2)
library(RColorBrewer)
library(tximport)
library(GenomicFeatures)
library(stringr)
library(openxlsx)
library(ape)
library(msigdbr)
library(fgsea)
library(enrichplot)
library(DOSE)
library(clusterProfiler)
library(stringr)
library(vegan)
library(tidyr)
library(tools)
library(ggpubr)  # For stat_cor
library(sva)

## Generate second metadata table

### Get DenseMatrix 

In [ ]:
# Define batches and conditions

batches <- paste0(rep("Batch", 12), rep(1:6, each=2))
conditions <- rep(c("unstimulated", "stimulated"), 6)

# Initialize dense_matrix list (list of count matrices for each plate and condition)
dense_matrixList <- list()

# Iterate over batches to fill dense_matrixList
for (i in 1:length(batches)) {
    sample = batches[i]
    condition = conditions[i]
    baseName = paste0(sample, '.', condition)
    
    fName = paste0('/rprojectnb/cancergrp/brb/intermediate_outputs/', baseName, '/out.Solo.out/Gene/raw')
    data <- Read10X(data.dir = fName)
    seurat_object <- CreateSeuratObject(counts = data)
    dense_matrix <- as.matrix(seurat_object@assays$RNA$counts)

    fName = paste0('/rprojectnb/cancergrp/brb/metadataBarcodes/', baseName, '.metadata.tsv')

    # *metadata is a temp object
    metadata = read_tsv(fName, col_names = c('Well', 'Type', 'BC'), show_col_types = FALSE)
    metadata = metadata %>% mutate(id = paste(baseName, Well, sep ='_'))
    colnames(dense_matrix) = metadata$id[match(colnames(dense_matrix), metadata$BC)]
    dense_matrixList[[baseName]] <- dense_matrix
    
}

# Merge list elements into a single matrix
DenseMatrix <- do.call(cbind, dense_matrixList)

In [ ]:
megaMetadata = read_excel('/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/brb_metadata.xlsx')

## Rename DenseMatrix column names
colnames(DenseMatrix) = megaMetadata$id2[match(colnames(DenseMatrix), megaMetadata$id)]

# Show a subset
DenseMatrix[1:3, 1:3]

In [ ]:
# New metadata object that summarizes stats from the first metadata table

metadata = data.frame(row.names = colnames(DenseMatrix))
metadata$batch_id = str_split_fixed(str_split_fixed(rownames(metadata), '_', 4)[,1], '\\.', 2)[,1]
metadata$condition = str_split_fixed(str_split_fixed(rownames(metadata), '_', 4)[,1], '\\.', 2)[,2]
metadata$well = str_split_fixed(rownames(metadata), '_', 4)[,2]
metadata$vtr = str_split_fixed(rownames(metadata), '_', 4)[,3]
metadata$replicate = str_split_fixed(rownames(metadata), '_', 4)[,4]
metadata$n_reads_all_genes = megaMetadata$n_total_counts[match(rownames(metadata), megaMetadata$id2)]
metadata$Match = megaMetadata$Match2[match(rownames(metadata), megaMetadata$id2)]
metadata$ID = rownames(metadata)
head(metadata)

In [ ]:
# Save second metadata table as xlsx file
write.xlsx(metadata, '/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/brb_metadata_counts.xlsx')

### Get tr2dg

In [ ]:
# Function that removes gene and transcripts name versions
remove_version <- function(names) {
  return(sub("\\..*", "", names))
}

tr2g <- read.gff('/rprojectnb/cancergrp/brb/annotation_files/gencode.v46.primary_assembly.basic.annotation.gff3', na.strings = c(".", "?"), GFF3 = TRUE)

tr2g <- tr2g[tr2g$type == 'transcript',]

gene_id <- str_split(tr2g$attributes, pattern = ';') %>% unlist() %>% str_subset('gene_id')
gene_id <- str_split_fixed(gene_id, '=', 2)[,2]

transcript_id <- str_split(tr2g$attributes, pattern = ';') %>% unlist() %>% str_subset('transcript_id')
transcript_id <- str_split_fixed(transcript_id, '=', 2)[,2]

gene_type <- str_split(tr2g$attributes, pattern = ';') %>% unlist() %>% str_subset('gene_type')
gene_type <- str_split_fixed(gene_type, '=', 2)[,2]

gene_name <- str_split(tr2g$attributes, pattern = ';') %>% unlist() %>% str_subset('gene_name')
gene_name <- str_split_fixed(gene_name, '=', 2)[,2]

tr2dg <- tibble(gene_id, gene_type, gene_name, transcript_id)
tr2dg$gene_id_nv <- remove_version(tr2dg$gene_id)


### DGE - DeSeq2 Analysis

In [ ]:
## Strategy 2 (All plates)

# Read second metadata table
metadata <- as.data.frame(read_excel('/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/brb_metadata_counts.xlsx'))

# Add new ID columns and convert data type to factor to be used in DESeq2 later
rownames(metadata) = metadata$ID
metadata$ID2 = paste(paste(metadata$batch_id, metadata$condition, sep = '.'), metadata$vtr, sep = '_')
metadata$ID2 = factor(metadata$ID2)
metadata$vtr = factor(metadata$vtr)

varianceList <- list()

dge_objects_folder0 <- '/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/strategies/'

# Loop for each condition and batch
for (i in unique(metadata$condition)) {
    batches <- unique(metadata$batch_id)[!unique(metadata$batch_id) %in% c('Batch1', 'Batch2')]

    # Iterate over conditions ('All', Batch3-6) - Batch1 and Batch2 are skipped due to low bad quality of reads
    print(i)
  
    ## Subsetting all samples that belong to a specific condition
    keepSamples <- rownames(metadata %>% filter(condition == i, batch_id %in% batches))
    subMatrix = DenseMatrix[, keepSamples]
    submetadata = metadata[keepSamples,]

    dge_objects_folder <- paste0(dge_objects_folder0, '2_All_Batches')

    #################################
    # first >>> batch_id <<< then >>> vtr <<< ! 
    design <- ~ batch_id + vtr
    #################################
                    
    ## Remove samples that correspond to Batch1 OR were not highly confident in terms of VTR reads mapping
    keepSamples <- rownames(submetadata %>% filter(Match == 'Yes'))
    subMatrix = subMatrix[, keepSamples]
    submetadata = submetadata[keepSamples,]
    dim(subMatrix)
    
    ## Remove samples with less than 500k counts
    n_counts = 0.5 * 10**6 
    keepSamples <- rownames(submetadata %>% filter(n_reads_all_genes > n_counts))
    subMatrix = subMatrix[, keepSamples]
    submetadata = submetadata[keepSamples,]
    dim(subMatrix)

    submetadata$vtr = factor(as.character(submetadata$vtr))
    
    print('Create DESeq2 ...')
    
    ## Create the DESeqDataSet with the filtered data
    
    dds <- DESeqDataSetFromMatrix(
      countData = subMatrix,
      colData = submetadata,
      design = design)
    dim(dds)
    
    ## Filter genes corresponding to a set of genetype
    set_genetype = c('protein_coding', 'TR_V_gene', 'TR_J_gene', 'TR_D_gene', 'TR_C_gene', 'IG_V_gene', 'IG_LV_gene', 'IG_J_gene', 'IG_D_gene', 'IG_C_gene')
    keep1 = rownames(dds) %in% tr2dg$gene_id_nv[tr2dg$gene_type  %in% set_genetype]
    keep2 = rownames(dds) %in% tr2dg$gene_name[tr2dg$gene_type  %in% set_genetype]
    keep = keep1 | keep2
    dds <- dds[keep, ]
    dim(dds)
    
    # Remove mitochondrial genes and ribosomal gens
    keep1 = rownames(dds) %in% tr2dg$gene_name[!str_detect(tr2dg$gene_name, '^MT-')]
    keep2 = rownames(dds) %in% tr2dg$gene_name[!str_detect(tr2dg$gene_name, '^RPS')]
    keep3 = rownames(dds) %in% tr2dg$gene_name[!str_detect(tr2dg$gene_name, '^RPL')]
    keep = keep1 & keep2 & keep3
    dds <- dds[keep, ]
    dim(dds)
                              
    ## Filter lowly expressed genes (lets try values)
    keepGenes <- rowMeans(counts(dds)) > 10 # Change to 12 or 0

    dds <- dds[keepGenes, ]
    
    # Normalize values
    dds <- estimateSizeFactors(dds)
    
    ############################################################
    transf.data <- varianceStabilizingTransformation(dds, blind = TRUE)
    variances <- rowVars(assay(transf.data))                  
    ############################################################

    
    dds$vtr <- relevel(dds$vtr, ref = 'vector')
    ds.deseq2 <- DESeq(dds)

    # Create the folder if it doesn't exist
    
    if (!dir.exists(dge_objects_folder)) {
      dir.create(dge_objects_folder, recursive = TRUE)
    }
    
    # Save the DESeq2 object
    
    fname <- paste0(dge_objects_folder, "/", j, "_", i, ".rds")
    saveRDS(ds.deseq2, file = fname)
    
    cat("DESeq2 object saved to:", fname, "\n")

    all_info = results(ds.deseq2,
                        altHypothesis = 'greaterAbs',
                        alpha = 0.1,
                        pAdjustMethod = "BH",
                        independentFiltering = FALSE)
    all_info = data.frame(all_info)
    all_info$symbol <- rownames(all_info)   
    
    temp_df <- data.frame(variances = variances, 
                          all_info[match(names(variances), rownames(all_info)),
                                    c("baseMean", "log2FoldChange", "padj")])
    
    varianceList[[i]][[j]] <- temp_df
}


In [ ]:
# --- Setup Paths ---
fileDir2 <- "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/strategies/2_All_Batches"
dge_tables_folder <- '/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/filtered_tables/'

# Create output directory once
if (!dir.exists(dge_tables_folder)) {
  dir.create(dge_tables_folder, recursive = TRUE)
}

all_plates <- list.files(path = fileDir2, pattern = "^All.*\\.rds$", full.names = TRUE)

# --- Loop Strategy 2 Only ---
for (rdsObject in all_plates) {
  
  dds <- readRDS(rdsObject)
  dds.count <- counts(dds, normalize = TRUE)
  
  # Determine Condition
  if (str_detect(rdsObject, "unstimulated")) {
    condition_id <- "unstimulated"
  } else if (str_detect(rdsObject, "stimulated")) {
    condition_id <- "stimulated"
  }
  
  # Get VTRs (Specific filter for Strategy 2/All Batches)
  vtrs <- metadata %>% 
    filter(condition == condition_id, Match == 'Yes', !batch_id %in% c('Batch1', 'Batch2')) %>% 
    pull(vtr) %>% 
    unique()
  
  vtrs <- vtrs[!vtrs %in% c("vector")]
  
  for (vtr_id in vtrs) {
    print(paste(vtr_id))
    
    ### Filter low expression genes ###
    cols <- grep(paste0("(^|[_\\.])", vtr_id, "($|[_\\.])"), colnames(dds.count), value = TRUE)
    
    keepGenes <- if (is.matrix(dds.count[, cols])) {
      rowMeans(dds.count[, cols]) > 10 
    } else {
      dds.count[, cols] > 10 
    }
    sub_dds <- dds[keepGenes, ]
    
    ### Run Results ###
    name_str <- paste0("vtr_", vtr_id, "_vs_vector")
    
    all_info <- as.data.frame(results(
      sub_dds,
      altHypothesis = "greaterAbs",
      alpha = 0.1,
      pAdjustMethod = "BH",
      independentFiltering = FALSE,
      name = name_str
    ))
    
    all_info$symbol <- rownames(all_info)
    
    # Generate Batch Name for filename
    batch_name <- metadata %>% 
      filter(condition == condition_id, 
             Match == 'Yes', 
             !batch_id %in% c('Batch1', 'Batch2'), 
             vtr == vtr_id) %>%
      pull(batch_id) %>% unique %>% paste(collapse = ".")
    
    extra_name <- paste0("_", batch_name)
    
    print(dim(dds))
    print(dim(sub_dds))
    
    # Save (Hardcoded strategy 2 in filename)
    file_path <- paste0(dge_tables_folder, vtr_id, "_", condition_id,
                        extra_name, "_strategy2.xlsx")
    
    write_xlsx(all_info, file_path)
  }
}